<a href="https://colab.research.google.com/github/esalinasbio/taller-modelado-biomolecular/blob/master/notebooks/02_Analisis_Trayectoria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 2 - Análisis de trayectoria

**Taller de Modelado Biomolecular**

---

En el ejercicio 1 preparamos y corrimos una simulación corta. Aquí trabajamos con **la misma simulación pero mucho más larga**, previamente calculada.

**Este ejercicio no necesita GPU.** Si la tienen asignada, cámbienla a CPU para liberarla.

| Bloque | Qué medimos | Pregunta |
|---|---|---|
| **A** | RMSD, RMSF, Rg de proteína y RNA | ¿Se comportan igual las dos moléculas? |
| **B** | eRMSD y pares de bases | ¿Es el RMSD la métrica correcta para RNA? |
| **C** | Contactos, puentes de H, apilamiento | ¿Qué sostiene el reconocimiento? |
| **D** | Análisis de componentes principales | ¿Cuáles son los movimientos colectivos? |
| **E** | Convergencia y comparación con RMN | ¿Le creemos a esto? |

---

#### **RECUERDA:** Presiona `Ctrl + S` o usa el menú `Archivo` (`File`) para guardar una copia del notebook a tu Google Drive

In [ ]:
#@title Instalar y cargar
import time; _t0 = time.time()
!pip install -q mdtraj py3Dmol barnaba 2>/dev/null
import numpy as np, matplotlib.pyplot as plt, mdtraj as mdt, py3Dmol
plt.rcParams.update({"figure.dpi": 100, "font.size": 11})
AZUL, NARANJA, GRIS, MORADO = "#2E8B8B", "#E8813A", "#64748b", "#7c3aed"
print(f"Listo en {time.time()-_t0:.0f} s")

In [ ]:
#@title Descargar y cargar la trayectoria
_BASE = ("https://raw.githubusercontent.com/esalinasbio/taller-modelado-biomolecular/"
         "6f79e4f95bd2fbb33bd31016bb89b8600ce28754/data/MD")
TRAYECTORIA = "1aud_100ns_noWAT.xtc"
TOPOLOGIA   = "1aud_topologia_noWAT.pdb"
PS_POR_CUADRO = 10

import os
for archivo in (TRAYECTORIA, TOPOLOGIA):
    if not os.path.exists(archivo):
        !wget -q --show-progress -O {archivo} {_BASE}/{archivo}
#!ls -lh {TRAYECTORIA} {TOPOLOGIA}

tr = mdt.load(TRAYECTORIA, top=TOPOLOGIA)
tiempo = np.arange(tr.n_frames)*PS_POR_CUADRO/1000.0
mitad = tr.n_frames//2

sel_prot = tr.topology.select("protein")
sel_rna  = tr.topology.select("not protein")
sel_ca   = tr.topology.select("protein and name CA")
sel_p    = tr.topology.select("not protein and name P")
res_prot = [r for r in tr.topology.residues if r.is_protein]
res_rna  = [r for r in tr.topology.residues if not r.is_protein]

print("="*56)
print(f"  Frames             : {tr.n_frames}")
print(f"  Tiempo total       : {tiempo[-1]:.1f} ns  (1 frame = {PS_POR_CUADRO} ps)")
print(f"  Átomos             : {tr.n_atoms}")
print(f"  Aminoácidos        : {len(res_prot)}")
print(f"  Nucleótidos        : {len(res_rna)}  ({''.join(r.name[-1] for r in res_rna)})")
print("="*56)

tr.superpose(tr, 0, atom_indices=sel_ca)


Alineamos sobre los Cα de la **PROTEÍNA**. Esa decisión no es neutral: todo lo que midamos del RNA será movimiento RELATIVO A LA PROTEÍNA, no movimiento interno del RNA. Si alineáramos sobre el RNA obtendríamos otros números, y ninguno de los dos sería "el correcto".

> Cuando lean un RMSD en un artículo, la primera pregunta es siempre:
>
> * **¿Cual es la referencia?**

---
# Bloque A - Métricas básicas

Antes de preguntar nada biológico hay que contestar tres preguntas técnicas sobre la trayectoria, y las tres métricas que aparecen en bastantes artículos de dinámica molecular. El **RMSD** es un número por frame: representa la desviación de las coordenadas atómicas respecto a una referencia  (*¿se tanto se alejó del inicio?*). El **RMSF** es un número por residuo: mide que tanto se mueve cada uno de los resiuos (*¿qué partes se mueven?*). El **Rg** (radio de giro) mide que tan compacta está la estructura (*¿sigue plegada?*).

Las calculamos por separado para la proteína y para el RNA

> **Pregunta:**
>
> **¿cuál de las dos dará un RMSF mayor, y por qué?**

In [ ]:
#@title A - RMSD, RMSF y radio de giro
rmsd_ca  = 10*mdt.rmsd(tr, tr, 0, atom_indices=sel_ca)
rmsd_rna = 10*mdt.rmsd(tr, tr, 0, atom_indices=sel_p)
rg_prot  = 10*mdt.compute_rg(tr.atom_slice(sel_prot))

def rmsf_de(sub_sel, sel_str):
    s = tr.atom_slice(sub_sel)
    idx = s.topology.select(sel_str)
    s.superpose(s, 0, atom_indices=idx)
    return 10*mdt.rmsf(s, s, 0, atom_indices=idx), idx, s

rmsf_ca, idx_ca_loc, s_prot = rmsf_de(sel_prot, "name CA")
rmsf_rna, idx_p_loc, s_rna  = rmsf_de(sel_rna,  "name P")
num_res = [s_prot.topology.atom(i).residue.resSeq for i in idx_ca_loc]
etq_nt  = [f"{s_rna.topology.atom(i).residue.name[-1]}{s_rna.topology.atom(i).residue.resSeq}"
           for i in idx_p_loc]

fig, ax = plt.subplots(2, 2, figsize=(12.5, 6.4))
ax[0,0].plot(tiempo, rmsd_ca, lw=0.8, color=AZUL, label="proteína (Cα)")
ax[0,0].plot(tiempo, rmsd_rna, lw=0.8, color=NARANJA, label="RNA (P)")
ax[0,0].set_xlabel("tiempo (ns)"); ax[0,0].set_ylabel("RMSD (Å)")
ax[0,0].set_title("La misma simulación, dos moléculas"); ax[0,0].legend(fontsize=8)

ax[0,1].plot(tiempo, rg_prot, lw=0.8, color=AZUL)
ax[0,1].set_xlabel("tiempo (ns)"); ax[0,1].set_ylabel("Rg (Å)")
ax[0,1].set_title("Radio de giro de la proteína")

ax[1,0].plot(num_res, rmsf_ca, lw=1.3, color=AZUL)
ax[1,0].fill_between(num_res, 0, rmsf_ca, color=AZUL, alpha=0.18)
ax[1,0].set_xlabel("residuo"); ax[1,0].set_ylabel("RMSF (Å)")
ax[1,0].set_title("Flexibilidad por aminoácido")

ax[1,1].bar(range(len(rmsf_rna)), rmsf_rna, color=NARANJA, alpha=0.85)
ax[1,1].axhline(rmsf_ca.mean(), ls="--", c=AZUL, lw=1.4,
                label=f"media proteína = {rmsf_ca.mean():.2f} Å")
ax[1,1].set_xticks(range(len(rmsf_rna)))
ax[1,1].set_xticklabels(etq_nt, rotation=90, fontsize=6.5)
ax[1,1].set_ylabel("RMSF (Å)"); ax[1,1].set_title("Flexibilidad por nucleótido")
ax[1,1].legend(fontsize=8)
for a in ax.ravel(): a.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"RMSD medio  proteína {rmsd_ca[mitad:].mean():5.2f} Å  |  RNA {rmsd_rna[mitad:].mean():5.2f} Å")
print(f"RMSF medio  proteína {rmsf_ca.mean():5.2f} Å  |  RNA {rmsf_rna.mean():5.2f} Å")
print(f"RMSF máximo proteína {rmsf_ca.max():5.2f} Å  |  RNA {rmsf_rna.max():5.2f} Å")
print(f"Rg proteína: {rg_prot[mitad:].mean():.2f} ± {rg_prot[mitad:].std():.2f} Å  (no se desplegó)")

---
# Bloque B - Por qué el RMSD es mala métrica para RNA

El RMSD mide desplazamiento de átomos en el espacio. En una molécula de RNA la orientación y conectividad de las bases dictan su función. RMSD funciona bien para proteínas, no tanto para RNA: todo se mueve mucho aunque cada par de bases siga exactamente igual.

El **eRMSD** compara la *geometría relativa entre bases* (distancia, orientación, apilamiento) en vez de solo coordenadas atómicas. Es casi insensible aal movimiento del *backbone* y muy sensible a que un par de bases se rompa.

In [ ]:
#@title B - eRMSD y persistencia de pares de bases
try:
    import barnaba as bb
    from collections import Counter

    s_rna.save("rna.xtc"); s_rna[0].save("rna_ref.pdb")
    ermsd = np.array(bb.ermsd("rna_ref.pdb", "rna.xtc", topology="rna_ref.pdb"))

    fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.6))
    ax[0].plot(tiempo, rmsd_rna/rmsd_rna.max(), lw=0.8, color=GRIS,
               label="RMSD cartesiano (norm.)")
    a2 = ax[0].twinx()
    a2.plot(tiempo, ermsd, lw=1.0, color=NARANJA, label="eRMSD")
    a2.axhline(0.7, ls="--", c="#b91c1c", lw=1.3)
    a2.set_ylabel("eRMSD", color=NARANJA)
    ax[0].set_xlabel("tiempo (ns)"); ax[0].set_ylabel("RMSD (norm.)", color=GRIS)
    ax[0].set_title("RMSD vs eRMSD"); ax[0].grid(alpha=0.25)

    stack, pairs, res = bb.annotate("rna.xtc", topology="rna_ref.pdb")
    cuenta = Counter()
    for pp, an in pairs:
        for (i, j), a in zip(pp, an):
            if a == "WCc":
                cuenta[(min(i,j), max(i,j))] += 1
    n = len(pairs)
    filas = sorted(cuenta.items(), key=lambda x: -x[1])
    etiquetas = [f"{res[i]}-{res[j]}" for (i,j), k in filas]
    valores = [100*k/n for (i,j), k in filas]
    ax[1].barh(range(len(valores)), valores, color=NARANJA, alpha=0.85)
    ax[1].set_yticks(range(len(valores)))
    ax[1].set_yticklabels(etiquetas, fontsize=7)
    ax[1].invert_yaxis(); ax[1].set_xlabel("frecuencia (%)")
    ax[1].set_title("Persistencia de pares"); ax[1].grid(alpha=0.25, axis="x")
    plt.tight_layout(); plt.show()

    print(f"eRMSD medio {ermsd.mean():.3f} · máximo {ermsd.max():.3f} · "
          f"frames > 0.7: {100*(ermsd>0.7).mean():.1f} %")

except Exception as e:
    print("barnaba no disponible:", e)

---
# Bloque C - Qué sostiene el reconocimiento

Los bloques anteriores midieron propiedades globales. Éste responde una pregunta
biológica: **¿por qué esta proteína reconoce esta secuencia y no otra?**

Un RRM reconoce RNA con tres mecanismos que vamos a medir por separado:

1. **Contactos** en general — quién está cerca de quién, y cuánto tiempo
2. **Puentes de hidrógeno** — la parte que lee la *identidad* de la base
3. **Apilamiento aromático** — anillos de Phe/Tyr/Trp contra las bases

El tercero es el que define a los RRM y el que una estructura estática no puede
cuantificar.

In [ ]:
# @title C1 - Mapa de contactos de la interfaz
#@markdown Quién está cerca de quién, y cuánto tiempo

UMBRAL_NM = 0.45
CORTE_PREFILTRO = 1.2

todos = [(rp.index, rr.index) for rp in res_prot for rr in res_rna]

muestra = tr[::max(1, tr.n_frames//20)]
d0, _ = mdt.compute_contacts(muestra, contacts=todos, scheme="closest-heavy")
cerca = d0.min(axis=0) < CORTE_PREFILTRO
pares = [p for p, ok in zip(todos, cerca) if ok]

dist, _ = mdt.compute_contacts(tr, contacts=pares, scheme="closest-heavy")
ocup_pares = (dist < UMBRAL_NM).mean(axis=0)

# reconstruir el mapa completo con ceros donde no se evaluó
ocup = np.zeros(len(todos))
ocup[np.where(cerca)[0]] = ocup_pares
M = ocup.reshape(len(res_prot), len(res_rna))

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={"width_ratios": [1.5, 1]})

im = ax[0].imshow(M, aspect="auto", cmap="magma_r", vmin=0, vmax=1)
ax[0].set_xticks(range(len(res_rna)))
ax[0].set_xticklabels([f"{r.name[-1]}{r.resSeq}" for r in res_rna], rotation=90, fontsize=7)
ax[0].set_ylabel("residuo de la proteína")
ax[0].set_xlabel("nucleótido")
ax[0].set_title(f"Ocupancia de contacto (< {UMBRAL_NM*10:.1f} Å)")
plt.colorbar(im, ax=ax[0], label="fracción de cuadros")

ax[1].hist(ocup[ocup > 0.02], bins=40, color=MORADO, alpha=0.85)
ax[1].set_xlabel("ocupancia")
ax[1].set_ylabel("nº de pares residuo-nucleótido")
ax[1].set_title("¿Cuántos contactos son permanentes?")
ax[1].grid(alpha=0.25, axis="y")

plt.tight_layout()
plt.show()

orden = np.argsort(ocup)[::-1]

mapa = {i: j for j, i in enumerate(np.where(cerca)[0])}

print(f"{'residuo':<12}{'nucleótido':<12}{'ocupancia':>10}{'d media (Å)':>13}")
print("-"*48)
orden = np.argsort(ocup)[::-1]
for k in orden[:12]:
    a = tr.topology.residue(todos[k][0])
    b = tr.topology.residue(todos[k][1])
    print(f"{a.name}{a.resSeq:<7d}{b.name[-1]}{b.resSeq:<10d}"
          f"{100*ocup[k]:>9.1f}%{10*dist[:, mapa[k]].mean():>12.2f}")

print(f"\nContactos con ocupancia > 90 % : {(ocup > 0.9).sum()}")
print(f"Contactos transitorios (30-70 %): {((ocup > 0.3) & (ocup < 0.7)).sum()}")


In [ ]:
#@title C2 - Puentes de hidrógeno proteína–RNA
FREQ_MIN = 0.10   #@param {type:"number"}

hb = mdt.baker_hubbard(tr, freq=FREQ_MIN, periodic=False)
es_prot = np.isin(hb, sel_prot)
# nos quedamos sólo con los que cruzan la interfaz (donante y aceptor en cadenas distintas)
cruza = es_prot[:, 0] != es_prot[:, 2]
hb = hb[cruza]

if len(hb):
    d_HA = mdt.compute_distances(tr, hb[:, [1, 2]], periodic=False)
    ang  = mdt.compute_angles(tr, hb, periodic=False)
    ocupa = ((d_HA < 0.25) & (ang > 2.0)).mean(axis=0)   # criterio Baker-Hubbard
    orden = np.argsort(ocupa)[::-1]

    fig, ax = plt.subplots(figsize=(12, max(3, 0.4*min(len(orden), 20))))
    top_n = orden[:20]
    etiquetas = []
    for k in top_n:
        dn = tr.topology.atom(hb[k, 0]); ac = tr.topology.atom(hb[k, 2])
        etiquetas.append(f"{dn.residue.name}{dn.residue.resSeq}:{dn.name} → "
                         f"{ac.residue.name}{ac.residue.resSeq}:{ac.name}")
    colores = [AZUL if tr.topology.atom(hb[k,0]).residue.is_protein else NARANJA
               for k in top_n]
    ax.barh(range(len(top_n)), 100*ocupa[top_n], color=colores, alpha=0.9)
    ax.set_yticks(range(len(top_n))); ax.set_yticklabels(etiquetas, fontsize=7)
    ax.invert_yaxis(); ax.set_xlabel("ocupancia (%)")
    ax.set_title("Puentes de hidrógeno a través de la interfaz")
    ax.grid(alpha=0.25, axis="x"); plt.tight_layout(); plt.show()

    print(f"Puentes de H detectados en la interfaz : {len(hb)}")
    print(f"   con ocupancia > 80 %                : {(ocupa>0.8).sum()}")
    print(f"   con ocupancia 20-80 % (transitorios): {((ocupa>0.2)&(ocupa<0.8)).sum()}")
    print("\nAzul = donante en la proteína | Naranja = donante en el RNA\n")

    # ¿cuáles tocan el borde de la base (lectura de secuencia) vs el esqueleto?
    ATOMOS_BASE = {"N1","C2","O2","N3","C4","O4","N4","C5","C6","N6","N7","C8","N9","O6","N2"}
    lee_base = 0
    for k in range(len(hb)):
        for col in (0, 2):
            at = tr.topology.atom(hb[k, col])
            if (not at.residue.is_protein) and at.name in ATOMOS_BASE:
                lee_base += 1; break
    print(f"Puentes que con las bases : {lee_base} de {len(hb)}")
else:
    print("No se detectaron puentes de H en la interfaz con esa frecuencia mínima.")
    print("Bajen FREQ_MIN y vuelvan a intentar.")

Ésta es la distinción clave del reconocimiento específico.

Un puente de hidrógeno al *backbone* fosfato-ribosa sujeta el RNA pero no distingue una secuencia de otra. Un puente a las bases nitrógenadas sí lee la identidad del nucleótido: es lo que hace que la proteína reconozca esa secuencia y no cualquier RNA.

Si casi todos los puentes fueran al esqueleto, estaríamos viendo unión
inespecífica.

> **¿Cuántos de los de arriba tocan átomos des bases?**

In [ ]:
#@title C3 - Apilamiento aromático
D_MAX_NM   = 0.55
ANG_MAX_GR = 35
ANILLOS_AA = {"PHE": ["CG","CD1","CD2","CE1","CE2","CZ"],
              "TYR": ["CG","CD1","CD2","CE1","CE2","CZ"],
              "TRP": ["CD2","CE2","CE3","CZ2","CZ3","CH2"],
              "HIS": ["CG","ND1","CD2","CE1","NE2"]}
PURINA    = ["N9","C8","N7","C5","C4","N3","C2","N1","C6"]
PIRIMIDINA= ["N1","C2","N3","C4","C5","C6"]

def anillos_de(residuos, tabla=None):
    out = []
    for r in residuos:
        if tabla is not None:
            nombres = tabla.get(r.name)
        else:
            base = r.name.strip()[-1]
            nombres = PURINA if base in ("A","G") else PIRIMIDINA
        if not nombres: continue
        idx = [a.index for a in r.atoms if a.name in nombres]
        if len(idx) >= 5:
            out.append((r, np.array(idx)))
    return out

ar_prot = anillos_de(res_prot, ANILLOS_AA)
ar_rna  = anillos_de(res_rna)
print(f"Anillos aromáticos en la proteína: {len(ar_prot)} · bases del RNA: {len(ar_rna)}")

def centro_y_normal(xyz, idx):
    p = xyz[:, idx, :]                       # (nframes, natoms, 3)
    cen = p.mean(axis=1)
    q = p - cen[:, None, :]
    # normal = vector singular de menor valor (eje perpendicular al plano)
    _, _, vt = np.linalg.svd(q, full_matrices=False)
    return cen, vt[:, 2, :]

datos = []
for rp, ip in ar_prot:
    cp, np_ = centro_y_normal(tr.xyz, ip)
    for rr, ir in ar_rna:
        cr, nr = centro_y_normal(tr.xyz, ir)
        d = np.linalg.norm(cp - cr, axis=1)
        if d.min() > D_MAX_NM: continue
        cos = np.abs((np_*nr).sum(axis=1)).clip(0, 1)
        ang = np.degrees(np.arccos(cos))
        apila = (d < D_MAX_NM) & (ang < ANG_MAX_GR)
        if apila.mean() > 0.02:
            datos.append((rp, rr, apila.mean(), d, ang))

datos.sort(key=lambda x: -x[2])
if datos:
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 3.8))
    etq = [f"{a.name}{a.resSeq}·{b.name[-1]}{b.resSeq}" for a, b, o, d, g in datos[:12]]
    ax[0].barh(range(len(etq)), [100*o for _,_,o,_,_ in datos[:12]], color=MORADO, alpha=0.9)
    ax[0].set_yticks(range(len(etq))); ax[0].set_yticklabels(etq, fontsize=8)
    ax[0].invert_yaxis(); ax[0].set_xlabel("frecuencia (%)")
    ax[0].set_title("Apilamiento aromático"); ax[0].grid(alpha=0.25, axis="x")

    a, b, o, d, g = datos[0]
    ax[1].scatter(10*d, g, s=4, alpha=0.35, color=MORADO)
    ax[1].axvline(10*D_MAX_NM, ls="--", c="#b91c1c", lw=1.2)
    ax[1].axhline(ANG_MAX_GR, ls="--", c="#b91c1c", lw=1.2)
    ax[1].set_xlabel("distancia entre residuos (Å)")
    ax[1].set_ylabel("ángulo entre planos (°)")
    ax[1].set_title(f"Geometría: {a.name}{a.resSeq} · {b.name[-1]}{b.resSeq}")
    ax[1].grid(alpha=0.25)
    plt.tight_layout(); plt.show()

    print(f"{'par':<22}{'% apilado':>11}{'d media (Å)':>13}{'ángulo (°)':>12}")
    print("-"*58)
    for a, b, o, d, g in datos[:12]:
        m = (d < D_MAX_NM) & (g < ANG_MAX_GR)
        print(f"{a.name}{a.resSeq} · {b.name[-1]}{b.resSeq:<12d}{100*o:>10.1f}%"
              f"{10*d[m].mean() if m.any() else np.nan:>12.2f}{g[m].mean() if m.any() else np.nan:>12.1f}")
else:
    print("No se detectó apilamiento con esos criterios. Aflojen D_MAX_NM o ANG_MAX_GR.")

El apilamiento base-aromático es el mecanismo característico de los RRM: las
bases del RNA se "acuestan" sobre los anillos de Phe/Tyr de la hoja-β, y la
interacción es de tipo π-π, no un puente de hidrógeno.

La gráfica de la derecha muestra por qué esto necesita dinámica: el par oscila
en un rango de distancias y ángulos. Una estructura experimental les da **UN** punto de esa nube. La simulación les da la distribución entera, y esto es lo que determina la entropía de la interacción.

In [ ]:
#@title C4 - Ver la interfaz en 3D
N_RESALTAR = 7   #@param {type:"integer"}

tr[0].save_pdb("frame0.pdb")
serial = {}
for linea in open("frame0.pdb"):
    if linea.startswith(("ATOM", "HETATM")):
        serial.setdefault(int(linea[22:26]), []).append(int(linea[6:11]))

destacados = [(tr.topology.residue(todos[k][0]), tr.topology.residue(todos[k][1]))
              for k in np.argsort(ocup)[::-1][:N_RESALTAR]]

v = py3Dmol.view(width=800, height=470)
v.addModel(open("frame0.pdb").read(), "pdb", {"keepH": True})
v.setStyle({}, {"cartoon": {"color": "#cbd5e1", "opacity": 0.6}})
for rp, rr in destacados:
    v.addStyle({"serial": serial.get(rp.resSeq, [])},
               {"stick": {"radius": 0.20, "colorscheme": "cyanCarbon"}})
    v.addStyle({"serial": serial.get(rr.resSeq, [])},
               {"stick": {"radius": 0.20, "colorscheme": "orangeCarbon"}})
v.zoomTo(); v.show()
print("Contactos de mayor ocupancia:")
for rp, rr in destacados:
    print(f"   {rp.name}{rp.resSeq} ··· {rr.name[-1]}{rr.resSeq}")

---
# Bloque D -  Componentes principales

Hasta aquí medimos cosas por residuo o por par. Pero una molécula no se mueve residuo por residuo: se mueve en **modos colectivos**, donde regiones enteras se desplazan de forma correlacionada.

El **análisis de componentes principales** (PCA) encuentra esos modos. Diagonaliza la matriz de covarianza de las posiciones atómicas y devuelve las direcciones del espacio conformacional que más variación explican. Las primeras dos o tres suelen capturar la mayor parte del movimiento.

Esto importa por dos razones:
- Primero, porque es la forma de contestar *¿qué hace el complejo?* en vez de cuánto se mueve cada residuo.
- Segundo, porque el espacio de las primeras componentes **es** un espacio conformacional de baja dimensión.

In [ ]:
#@title D1 - PCA sobre los átomos de esqueleto
SEL_PCA = "name CA or name P"   #@param ["name CA or name P", "name CA", "name P"]

idx_pca = tr.topology.select(SEL_PCA)
t_pca = tr.atom_slice(idx_pca)
t_pca.superpose(t_pca, 0)

X = t_pca.xyz.reshape(t_pca.n_frames, -1)      # (cuadros, 3N)
X = X - X.mean(axis=0)
cov = np.cov(X, rowvar=False)
val, vec = np.linalg.eigh(cov)
val, vec = val[::-1], vec[:, ::-1]
var_exp = 100*val/val.sum()
proj = X @ vec

print(f"Átomos en el PCA: {len(idx_pca)}  ->  {3*len(idx_pca)} dimensiones")
print(f"Varianza explicada:  PC1 {var_exp[0]:.1f} %  ·  PC2 {var_exp[1]:.1f} %  ·  "
      f"PC3 {var_exp[2]:.1f} %")
print(f"Las primeras 3 componentes explican {var_exp[:3].sum():.1f} % del movimiento total.")

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
# Modificado para mostrar la varianza explicada acumulada usando np.cumsum()
ax[0].plot(range(1, 21), np.cumsum(var_exp)[:20], "o-", color=MORADO, ms=4)
ax[0].set_xlabel("componente"); ax[0].set_ylabel("varianza acumulada (%)")
ax[0].set_title("Espectro"); ax[0].grid(alpha=0.25)

sc = ax[1].scatter(proj[:,0], proj[:,1], c=tiempo, cmap="viridis", s=6)
ax[1].set_xlabel(f"PC1 ({var_exp[0]:.0f} %)"); ax[1].set_ylabel(f"PC2 ({var_exp[1]:.0f} %)")
ax[1].set_title("Paisaje conformacional"); ax[1].grid(alpha=0.25)
plt.colorbar(sc, ax=ax[1], label="tiempo (ns)")

ax[2].plot(tiempo, proj[:,0], lw=0.8, color=MORADO, label="PC1")
ax[2].plot(tiempo, proj[:,1], lw=0.8, color=NARANJA, alpha=0.8, label="PC2")
ax[2].set_xlabel("tiempo (ns)"); ax[2].set_ylabel("proyección (nm)")
ax[2].set_title("Evolución temporal"); ax[2].legend(fontsize=8); ax[2].grid(alpha=0.25)
plt.tight_layout(); plt.show()

Cómo leer el panel del centro:

> Cada punto es un frame de la trayectoria, coloreado por tiempo. Si los colores están mezclados, el sistema visitó las mismas regiones una y otra vez.

Si ven dos o más manchas separadas, eso son estados conformacionales distintos.

In [ ]:
#@title D2 - ¿Qué movimiento ES la PC1?
COMPONENTE = 1   #@param {type:"integer"}
k = COMPONENTE - 1

# amplitud del modo por átomo: cuánto se mueve cada uno a lo largo de esta PC
modo = vec[:, k].reshape(-1, 3)
amplitud = 10*np.linalg.norm(modo, axis=1)*np.sqrt(val[k])*10   # Å aprox
etq = []
for i in idx_pca:
    a = tr.topology.atom(i)
    etq.append(f"{a.residue.name[-1] if not a.residue.is_protein else a.residue.name}"
               f"{a.residue.resSeq}")
es_rna_pca = np.array([not tr.topology.atom(i).residue.is_protein for i in idx_pca])

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.bar(np.where(~es_rna_pca)[0], amplitud[~es_rna_pca], color=AZUL, label="proteína")
ax.bar(np.where(es_rna_pca)[0], amplitud[es_rna_pca], color=NARANJA, label="RNA")
ax.set_xlabel("átomo (proteína | RNA)"); ax.set_ylabel("amplitud en PC%d (Å)" % COMPONENTE)
ax.set_title(f"Qué se mueve en la componente {COMPONENTE} "
             f"({var_exp[k]:.0f} % de la varianza)")
ax.legend(fontsize=8); ax.grid(alpha=0.25, axis="y"); plt.tight_layout(); plt.show()

print(f"Contribución a PC{COMPONENTE}:  proteína {100*np.sum(amplitud[~es_rna_pca]**2)/np.sum(amplitud**2):.0f} %"
      f"  ·  RNA {100*np.sum(amplitud[es_rna_pca]**2)/np.sum(amplitud**2):.0f} %")
print("\nÁtomos con mayor amplitud en este modo:")
for i in np.argsort(amplitud)[::-1][:10]:
    print(f"   {etq[i]:<10s} {amplitud[i]:.2f} Å")

# --- estructuras extremas del modo, para animar ---
ext = []
for s in np.linspace(-2, 2, 11):
    xyz = (X.mean(axis=0) + s*np.sqrt(val[k])*vec[:, k]).reshape(1, -1, 3)
    ext.append(xyz)
t_modo = mdt.Trajectory(np.concatenate(ext, axis=0) + t_pca.xyz.mean(axis=0),
                        t_pca.topology)
t_modo.save_pdb("modo_pc.pdb")

v = py3Dmol.view(width=760, height=430)
v.addModelsAsFrames(open("modo_pc.pdb").read(), "pdb")
v.setStyle({}, {"sphere": {"radius": 0.35, "color": "#94a3b8"}})
v.animate({"loop": "backAndForth", "interval": 90})
v.zoomTo(); v.show()
print(f"\nMovimiento exagerado a lo largo de PC{COMPONENTE} (esferas = Cα y P).")

Esto **NO** es una trayectoria: es una interpolación lineal a lo largo de un modo, amplificada para que se vea. Sirve para entender la DIRECCIÓN del movimiento, no su cinética.

---
# Bloque E - ¿Le creemos a esto?

Dos comprobaciones. La primera es interna: ¿muestreó lo suficiente? La segunda es
externa: ¿coincide con lo que dice el experimento?

In [ ]:
#@title E1 - Convergencia: primera mitad contra segunda
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))

for j, (sub_sel, sel_str, nombre, color) in enumerate([
        (sel_prot, "name CA", "proteína", AZUL),
        (sel_rna,  "name P",  "RNA",      NARANJA)]):
    s = tr.atom_slice(sub_sel); idx = s.topology.select(sel_str)
    a_, b_ = s[:mitad], s[mitad:]
    a_.superpose(a_, 0, atom_indices=idx); b_.superpose(b_, 0, atom_indices=idx)
    ra = 10*mdt.rmsf(a_, a_, 0, atom_indices=idx)
    rb = 10*mdt.rmsf(b_, b_, 0, atom_indices=idx)
    r = np.corrcoef(ra, rb)[0, 1]
    ax[j].plot(ra, lw=1.3, color=color, label="1a mitad")
    ax[j].plot(rb, lw=1.3, color=color, ls="--", alpha=0.75, label="2a mitad")
    ax[j].set_title(f"{nombre} · r = {r:.3f}"); ax[j].set_xlabel("residuo")
    ax[j].set_ylabel("RMSF (Å)"); ax[j].legend(fontsize=8); ax[j].grid(alpha=0.25)
    print(f"{nombre:10s} correlación entre mitades = {r:.3f}")

for serie, nombre, color in [(rmsd_ca, "proteína", AZUL), (rmsd_rna, "RNA", NARANJA)]:
    acum = np.cumsum(serie)/np.arange(1, len(serie)+1)
    ax[2].plot(tiempo, acum, lw=1.4, color=color, label=nombre)
    deriva = 100*abs(acum[-1]-acum[int(0.8*len(acum))])/acum[-1]
    print(f"{nombre:10s} deriva en el último 20 % = {deriva:.2f} %")
ax[2].set_xlabel("tiempo (ns)"); ax[2].set_ylabel("RMSD acumulado (Å)")
ax[2].set_title("Promedio acumulado"); ax[2].legend(fontsize=8); ax[2].grid(alpha=0.25)
plt.tight_layout(); plt.show()

Correlación cercana a 1 entre mitades: las dos ven el mismo patrón de
flexibilidad. Deriva menor a ~1 % en el último quinto: el promedio ya se
estabilizó.

Las dos son condiciones NECESARIAS, no suficientes. Una trayectoria atrapada
en un solo mínimo se ve perfectamente convergida y está mal. La única defensa
real son las réplicas independientes.

In [ ]:
#@title E2 - La simulación contra el ensamble de RMN
!wget -q https://files.rcsb.org/download/1AUD.pdb -O 1AUD.pdb
ens = mdt.load("1AUD.pdb")

def orden_canonico(t):
    p = sorted(t.topology.select("name P"),
               key=lambda i: t.topology.atom(i).residue.resSeq)
    ca = sorted(t.topology.select("name CA"),
                key=lambda i: t.topology.atom(i).residue.resSeq)
    return np.array(p + ca), len(p)

i_ens, n_p_ens = orden_canonico(ens)
i_md,  n_p_md  = orden_canonico(tr)
print(f"Átomos comparables — ensamble: {len(i_ens)} · simulación: {len(i_md)}")

if len(i_ens) == len(i_md):
    e = ens.atom_slice(i_ens); e.superpose(e, 0)
    disp = 10*mdt.rmsf(e, e, 0)
    m = tr.atom_slice(i_md); m.superpose(m, 0)
    rmsf_md = 10*mdt.rmsf(m, m, 0)

    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    x = np.arange(len(disp))
    ax[0].plot(x, rmsf_md, lw=1.4, color=MORADO, label=f"RMSF simulación ({tiempo[-1]:.0f} ns)")
    ax[0].plot(x, disp, lw=1.4, color=GRIS, ls="--",
               label=f"dispersión ensamble RMN ({ens.n_frames} modelos)")
    ax[0].axvline(n_p_md-0.5, color="#b91c1c", lw=1)
    ax[0].text(n_p_md*0.4, ax[0].get_ylim()[1]*0.9, "RNA", fontsize=9, color=NARANJA)
    ax[0].text(n_p_md*1.4, ax[0].get_ylim()[1]*0.9, "proteína", fontsize=9, color=AZUL)
    ax[0].set_xlabel("átomo (P del RNA | Cα de la proteína)"); ax[0].set_ylabel("Å")
    ax[0].legend(fontsize=8); ax[0].grid(alpha=0.25)
    ax[0].set_title("¿Coinciden los patrones de flexibilidad?")

    ax[1].scatter(disp, rmsf_md, s=18, alpha=0.7,
                  c=["#E8813A"]*n_p_md + ["#2E8B8B"]*(len(disp)-n_p_md))
    lim = max(disp.max(), rmsf_md.max())*1.05
    ax[1].plot([0, lim], [0, lim], ls="--", c=GRIS, lw=1)
    ax[1].set_xlim(0, lim); ax[1].set_ylim(0, lim)
    ax[1].set_xlabel("dispersión del ensamble RMN (Å)")
    ax[1].set_ylabel("RMSF de la simulación (Å)")
    ax[1].set_title(f"r = {np.corrcoef(disp, rmsf_md)[0,1]:.3f}"); ax[1].grid(alpha=0.25)
    plt.tight_layout(); plt.show()

    print(f"RMSF medio simulación: {rmsf_md.mean():.2f} Å  |  ensamble RMN: {disp.mean():.2f} Å")
else:
    print("Los conjuntos de átomos no coinciden; revisar el orden de cadenas.")

---

## Cierre

- **Proteína:** plegada, estable, converge rápido. El caso fácil.
- **RNA:** más flexible, converge mucho más despacio, y el RMSD no es la métrica adecuada, para eso está el eRMSD.
- **Interfaz:** un núcleo pequeño de contactos permanentes rodeado de contactos
  transitorios. Los puentes de hidrógeno a las bases son los que leen
  la secuencia, los del esqueleto sólo sujetan. Y el apilamiento aromático.
- **PCA:** el movimiento no ocurre residuo por residuo sino en modos colectivos, y esos modos definen un espacio conformacional de pocas dimensiones.
- **Convergencia:** se comprueba, no se supone

---

### Tres preguntas para llevarse

1. Si mañana les llega un artículo con una simulación de 50 ns de un complejo proteína–RNA y una sola réplica, ¿qué le preguntarían a los autores?

2. Elijan un contacto con ocupancia intermedia (40–70 %). Si tuvieran que diseñar un experimento de mutagénesis, ¿por qué elegirían ese y no uno de los que están al 99 %?

3. Su propia molécula: ¿qué observable tendrían que medir, y cuánto tiempo
   necesitarían para que ese observable convergiera?